# Sentiment Score Features

## Overview

This notebook demonstrates the public functions in `sentiment_score_features` using compact synthetic news.
- Problem: raw article text cannot be used directly as a numeric feature in a tabular model.
- Approach: classify synthetic headlines and summaries into positive, negative, and neutral probabilities, then calculate a signed sentiment score.
- Sentiment Score Features: It produces article-level probabilities and a score where positive values indicate favorable language.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.data_preprocessing.sentiment_score_features import score_sentiment_features

## Score Synthetic News

This cell defines synthetic articles and deterministic three-class classifier outputs.
- The articles use clearly positive, negative, and neutral language so score direction is interpretable.
- `synthetic_finbert_classifier` follows the FinBERT output contract without downloading model weights.
- `created_at` and `symbols` remain available to align scores with market observations.

In [ ]:
news = pd.DataFrame(
    {
        "created_at": pd.date_range("2025-01-02", periods=3, freq="D", tz="UTC"),
        "symbols": ["AAPL", "AAPL", "AAPL"],
        "headline": [
            "Company reports record revenue",
            "Company cuts outlook after weak demand",
            "Company schedules annual shareholder meeting",
        ],
        "summary": [
            "Profit exceeded analyst expectations.",
            "Management expects lower sales next quarter.",
            "The meeting date and voting process were announced.",
        ],
    },
)

synthetic_predictions = [
    [
        {"label": "positive", "score": 0.88},
        {"label": "negative", "score": 0.04},
        {"label": "neutral", "score": 0.08},
    ],
    [
        {"label": "positive", "score": 0.03},
        {"label": "negative", "score": 0.91},
        {"label": "neutral", "score": 0.06},
    ],
    [
        {"label": "positive", "score": 0.08},
        {"label": "negative", "score": 0.07},
        {"label": "neutral", "score": 0.85},
    ],
]

def synthetic_finbert_classifier(texts, **kwargs):
    return synthetic_predictions[:len(texts)]

sentiment_features = score_sentiment_features(
    news,
    classifier=synthetic_finbert_classifier,
)

assert sentiment_features.loc[0, "sentiment_score"] > 0
assert sentiment_features.loc[1, "sentiment_score"] < 0
assert abs(sentiment_features.loc[2, "sentiment_score"]) < 0.05
display(sentiment_features)

This cell visualizes the synthetic sentiment scores.
- The favorable article should have a positive score, while the unfavorable article should have a negative score.
- The meeting announcement should remain close to zero because its neutral probability dominates.

In [ ]:
ax = sentiment_features.plot.bar(
    x="headline",
    y="sentiment_score",
    legend=False,
    color=["tab:green", "tab:red", "tab:gray"],
    rot=20,
)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Positive probability minus negative probability")
ax.set_title("Synthetic FinBERT Sentiment Scores")
plt.tight_layout()
plt.show()